In [1]:
import numpy as np
import torch
from parameter import Parameter
from test_tools import compare_grads, timing, tensor_to_parameter

In [2]:
%load_ext autoreload
%autoreload 1
%aimport test_tools
%aimport parameter

In [3]:
b = Parameter(np.random.rand(4, 4))
b

Parameter(shape = (4, 4) | size = 16 | label = NaN)

## example workflow 1

In [4]:
x1 = torch.rand(1, 4); x1.requires_grad = True
x2 = torch.rand(1, 4); x2.requires_grad = True

x3 = torch.rand(4, 1); x3.requires_grad = True
x4 = torch.rand(4, 1); x4.requires_grad = True

o_1 = x1 / x2
o_2 = x3 * x4

o_3 = o_1 @ o_2
o_3.backward()

print(x1.grad)
print(x2.grad)
print(x3.grad)
print(x4.grad)

tensor([[0.9550, 0.2066, 0.3389, 0.0844]])
tensor([[-1.5768, -0.1864, -0.2272, -0.2921]])
tensor([[1.1731],
        [0.3252],
        [0.1195],
        [1.8126]])
tensor([[0.8257],
        [0.4199],
        [0.6652],
        [0.0583]])


In [5]:
x1_ = Parameter(x1.clone().detach().numpy(), label='x1')
x2_ = Parameter(x2.clone().detach().numpy(), label='x2')
x3_ = Parameter(x3.clone().detach().numpy(), label='x3')
x4_ = Parameter(x4.clone().detach().numpy(), label='x4')

o_1_ = x1_ / x2_        ;o_1_._label = 'o_1'
o_2_ = x3_ * x4_        ;o_2_._label = 'o_2'
o_3_ = o_1_ @ o_2_      ;o_3_._label = 'o_3'

o_3_.backward()

print(x1_.grad)
print(x2_.grad)
print(x3_.grad)
print(x4_.grad)

[[0.95504231 0.20664187 0.33886842 0.08439818]]
[[-1.57675913 -0.18641363 -0.227239   -0.29207705]]
[[1.17312176]
 [0.32524012]
 [0.11953785]
 [1.81257196]]
[[0.82572987]
 [0.41988111]
 [0.66517402]
 [0.05830788]]


## 1 neuron test

In [6]:
BATCH_SIZE = 128

# Linear(in_features=784, out_features=10)
w1 = torch.empty(784, 10); w1.requires_grad = True
b1 = torch.zeros(BATCH_SIZE, 10); b1.requires_grad = True

# Initiate
torch.nn.init.xavier_uniform_(w1)

# softmax
softmax = torch.nn.Softmax(dim=-1)

# Loss function
loss_fn = torch.nn.NLLLoss()

# input data
x = torch.rand(BATCH_SIZE, 784)
y = torch.randint(0, 10, (BATCH_SIZE,))

# calculations
o_1 = x @ w1 + b1
logits = torch.log(softmax(o_1))
loss = loss_fn(logits, y)

# backward 
loss.backward()

In [7]:
a = torch.rand(128, 10)
a.sum(-1).shape

torch.Size([128])

In [8]:
BATCH_SIZE = 128
NUM_CLASSES = 10

# Linear(in_features=784, out_features=10)
W1 = tensor_to_parameter(w1, label='W1')
B1 = tensor_to_parameter(b1, label='B1')

# softmax
def Softmax(x: Parameter, dim=-1) -> Parameter:
    try:
        exp_x = x.exp();        exp_x._label='exp'

    except Exception as e:
        print("-" * 100)
        print(f"x.exp() operation failed:")
        print(f"- {e}")
        print(f"  - x.shape={x.shape}")
        print("-" * 100)
        raise
    
    try:
        sum_ = exp_x.sum(dim);  sum_._label='sum'
    except Exception as e:
        print("-" * 100)
        print(f"exp_x.sum({dim=}) operation failed:")
        print(f"- {e}")
        print(f"  - exp_x.shape={exp_x.shape}")
        print("-" * 100)
        raise

    try:
        out = exp_x / sum_
    except Exception as e:
        print("-" * 100)
        print(f"exp_x / sum_ operation failed:")
        print(f"- {e}")
        print(f"  - exp_x.shape={exp_x.shape}")
        print(f"  - sum_.shape={sum_.shape}")
        print("-" * 100)
        raise
    return out

# Loss function
def cross_entropy(pred, target):
    # pred: softmax output, shape (num_classes,)
    # target: one-hot, shape (num_classes,)

    def to_ohe(y, num_classes):
        out = np.zeros(num_classes)
        out[y.data] = 1.0
        return Parameter(out, label='ohe_Y')

    log_pred = pred.log()
    return -(to_ohe(target, NUM_CLASSES) * log_pred).sum()

# input data
X = tensor_to_parameter(x, label='X')
Y = tensor_to_parameter(y, label='Y')

# calculations
O_1 = X @ W1 + B1;                  O_1._label='O_1'
Logits = Softmax(O_1);              Logits._label='Logits' 
Loss = cross_entropy(Logits, Y);    Loss._label='Loss'

# backward 
Loss.backward()

In [9]:
print(logits)
print(Logits)

tensor([[-4.1835, -2.8738, -2.8901,  ..., -3.2597, -1.3639, -1.6444],
        [-4.6780, -3.2431, -1.9013,  ..., -2.8838, -1.9775, -2.1191],
        [-4.5029, -2.5580, -1.6904,  ..., -2.7437, -1.9954, -2.5637],
        ...,
        [-3.7600, -2.7735, -2.1315,  ..., -3.0817, -1.6627, -2.3500],
        [-4.5735, -3.1828, -1.4208,  ..., -2.5518, -1.6299, -1.8958],
        [-5.0574, -3.0765, -2.4203,  ..., -2.2823, -1.2843, -2.3304]],
       grad_fn=<LogBackward0>)
Parameter(shape = (128, 10) | size = 1280 | label = Logits)


## test 1

### parameters

In [10]:
"""TORCH"""
# Linear 1 (784, 64) (assuming batch_size=1)
w1 = torch.empty(784, 64); w1.requires_grad = True
b1 = torch.rand(1, 64); b1.requires_grad = True

# Linear 2 (64, 10)
w2 = torch.empty(64, 10); w2.requires_grad = True
b2 = torch.rand(1, 10); b2.requires_grad = True

# softmax
sm = torch.nn.Softmax(dim=-1)

# Loss function
loss_fn = torch.nn.NLLLoss()

torch.nn.init.xavier_uniform_(w1)
torch.nn.init.xavier_uniform_(w2)

x = torch.rand(1, 784)
target = torch.tensor([4])


"""CUSTOM"""
w1_ = Parameter(w1.detach().numpy(), label='w1')
b1_ = Parameter(b1.detach().numpy(), label='b1')
w2_ = Parameter(w2.detach().numpy(), label='w2')
b2_ = Parameter(b2.detach().numpy(), label='b2')
num_classes = 10

def sm_(x: Parameter, dim=-1) -> Parameter:
    exp_x = x.exp();        exp_x._label='exp'
    sum_ = exp_x.sum(dim);  sum_._label='sum'
    out = exp_x / sum_
    return out

def to_ohe(y, num_classes):
    out = np.zeros(num_classes)
    out[y.data] = 1.0
    return Parameter(out)

def cross_entropy(pred, target):
    # pred: softmax output, shape (num_classes,)
    # target: ohe, shape (num_classes,)
    log_pred = pred.log()
    return -(to_ohe(target, num_classes) * log_pred).sum()


x_ = Parameter(x.clone().numpy(), label='x')
target_ = Parameter(target.clone().numpy(), label='targets')

### calculations

In [11]:
"""TORCH"""
with timing('TORCH'):
    o_1 = x @ w1 + b1
    o_2 = o_1 @ w2 + b2
    
    o_sm = torch.log(sm(o_2))
    o_l = loss_fn(o_sm, target)
    
    o_l.backward()


"""CUSTOM"""
with timing('PARAMETER CLASS'):
    o_1_ = x_ @ w1_;      o_1_._label = 'o_1_'
    o_2_ = o_1_ + b1_;    o_2_._label = 'o_2_'
    o_3_ = o_2_ @ w2_;    o_3_._label = 'o_3_'
    o_4_ = o_3_ + b2_;    o_4_._label = 'o_4_'
    
    o_sm_ = sm_(o_4_).squeeze();       o_sm_._label = 'o_sm'
    o_l_ = cross_entropy(o_sm_, target_);     o_l_._label= 'o_l'
    
    o_l_.backward()

TORCH: 0.00023175s
PARAMETER CLASS: 0.00035813s


### output comparison

In [12]:
"""w_1"""
compare_grads(w1_, w1, 'w1')
compare_grads(w2_, w2, 'w2')
compare_grads(b1_, b1, 'b1')
compare_grads(b2_, b2, 'b2')

w1: max_diff=1.12e-07 ✓
w2: max_diff=6.03e-07 ✓
b1: max_diff=9.97e-08 ✓
b2: max_diff=2.19e-07 ✓


## \main focus

## looking for error 1

In [13]:
a1 = torch.tensor(4.0)
a2 = torch.tensor(2.0)

for i in [a1, a2]:
    i.requires_grad = True

out = a1 / a2
out.backward()

print(f"{out=}")
print(f"{a1.grad=}")
print(f"{a2.grad=}")

out=tensor(2., grad_fn=<DivBackward0>)
a1.grad=tensor(0.5000)
a2.grad=tensor(-1.)


In [14]:
a1_ = Parameter(4.0)
a2_ = Parameter(2.0)

out_ = a1_ / a2_
out_.backward()

print()
print(f"{out_.data=}")
print(f"{a1_.grad=}")
print(f"{a2_.grad=}")


out_.data=array(2.)
a1_.grad=array(0.5)
a2_.grad=array(-1.)


## looking for error 2

In [15]:
a1 = torch.tensor([2.0, 3.0, 4.0])
a2 = torch.tensor(2.0)

for i in [a1, a2]:
    i.requires_grad = True

out = a1.sum() ** a2
out.backward()

print(f"{out=}")
print(f"{a1.grad=}")
print(f"{a2.grad=}")

out=tensor(81., grad_fn=<PowBackward1>)
a1.grad=tensor([18., 18., 18.])
a2.grad=tensor(177.9752)


In [16]:
a1_ = Parameter([2.0, 3.0, 4.0])
a2_ = Parameter(2.0)

out_ = a1_.sum() ** 2.0
out_.backward()

print()
print(f"{out_.data=}")
print(f"{a1_.grad=}")
print(f"{a2_.grad=}")


out_.data=array([81.])
a1_.grad=array([18., 18., 18.])
a2_.grad=array(0.)
